\## a. Create and Index the Materialized Table

\### Step 1

```sql
-- =============================================
-- Step 1: Create Materialized Table with Surrogate Key
-- =============================================

IF OBJECT_ID('dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE', 'U') IS NOT NULL
BEGIN
    DROP TABLE dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE;
    PRINT 'Existing table dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE dropped.';
END

CREATE TABLE dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE (
    RecordID            INT IDENTITY(1,1) NOT NULL PRIMARY KEY, -- Surrogate Primary Key
    STUDY_ID           NVARCHAR(50) NOT NULL,
    POSTAL_CODE        NVARCHAR(20)  NULL,
    STREET_LINE        NVARCHAR(255)  NULL,
    EFF_DATE           DATE NOT NULL,
    END_DATE           DATE NOT NULL
);
GO

PRINT 'Table dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE created successfully with RecordID as primary key.';
```

## \### Step 2

In [6]:
-- =============================================
-- Step 2: Create Indexes on Materialized Table
-- =============================================

CREATE NONCLUSTERED INDEX IX_MT_FCT_COMBINED_HEALTH_CLIENT_STUDYID_EFFDATE
ON dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE (STUDY_ID, EFF_DATE);
GO

PRINT 'Index IX_MT_FCT_COMBINED_HEALTH_CLIENT_STUDYID_EFFDATE created successfully.';

CREATE NONCLUSTERED INDEX IX_MT_FCT_COMBINED_HEALTH_CLIENT_POSTAL_STREET
ON dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE (POSTAL_CODE, STREET_LINE);
GO

PRINT 'Index IX_MT_FCT_COMBINED_HEALTH_CLIENT_POSTAL_STREET created successfully.';

Commands completed successfully.

Index IX_MT_FCT_COMBINED_HEALTH_CLIENT_STUDYID_EFFDATE created successfully.

Index IX_MT_FCT_COMBINED_HEALTH_CLIENT_POSTAL_STREET created successfully.

Total execution time: 00:00:00.026

# \## b. Populate the Materialized Table

```sql
-- =============================================
-- Step 3: Populate Materialized Table
-- =============================================

BEGIN TRY
    BEGIN TRANSACTION;

    PRINT 'Starting data population into dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE...';

    INSERT INTO dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE (
        effective_year,
        effective_month,
        effective_day,
        STUDY_ID,
        POSTAL_CODE,
        CITY,
        STREET_LINE,
        LHA,
        CHSA,
        LATITUDE,
        LONGITUDE,
        EFF_DATE,
        END_DATE
    )
    SELECT
        effective_year,
        effective_month,
        effective_day,
        STUDY_ID,
        POSTAL_CODE,
        CITY,
        STREET_LINE,
        LHA,
        CHSA,
        LATITUDE,
        LONGITUDE,
        EFF_DATE,
        END_DATE
    FROM dev.VIEW_FCT_COMBINED_HEALTH_TABLE;

    COMMIT TRANSACTION;

    PRINT 'Data population completed successfully.';
END TRY
BEGIN CATCH
    IF @@TRANCOUNT > 0
        ROLLBACK TRANSACTION;

    PRINT 'An error occurred during data population.';
    PRINT ERROR_MESSAGE();
END CATCH
GO
```


\*Performance Considerations:\*

  

Batch Inserts: For extremely large datasets, consider inserting data in smaller batches to manage transaction log growth and reduce locking contention.

-- =============================================
-- Step 3: Populate Materialized Table
-- =============================================
-- Example: Batch Insertion (Adjust @BatchSize as needed)
DECLARE @BatchSize INT = 100000;
WHILE (1=1)
BEGIN
    BEGIN TRANSACTION;

    INSERT INTO dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE (
        STUDY_ID,
        POSTAL_CODE,
        STREET_LINE,
        EFF_DATE,
        END_DATE
    )
    SELECT TOP (@BatchSize)
    STUDY_ID,
    POSTAL_CODE,
    STREET_LINE,
    CAST(EFF_DATE AS DATE) AS EFF_DATE,
    CAST(END_DATE AS DATE) AS END_DATE
    FROM [HealthFiles_test].[dev].[FCT_COMBINED_HEALTH_CLIENT_TABLE] ct
    WHERE NOT EXISTS (
        SELECT 1
        FROM dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE mt
        WHERE mt.STUDY_ID = ct.STUDY_ID
          AND mt.EFF_DATE = ct.EFF_DATE
          AND mt.POSTAL_CODE = ct.POSTAL_CODE
          AND mt.STREET_LINE = ct.STREET_LINE
    );

    IF @@ROWCOUNT < @BatchSize
    BEGIN
        COMMIT TRANSACTION;
        BREAK;
    END

    COMMIT TRANSACTION;
END


3\. Implementing Incremental Refresh with Batch Processing

To handle large datasets efficiently and accommodate monthly data appends, we'll refactor the incremental refresh logic to:

  

Incorporate Additional Join Conditions: Use (STUDY\_ID, EFF\_DATE, POSTAL\_CODE) to accurately identify new records.

Implement Batch Processing: Insert data in manageable chunks to optimize performance and resource utilization.

Automate the Process: Use a stored procedure that can be scheduled via SQL Server Agent or Azure Automation.

a. Refactored Stored Procedure for Incremental Refresh

-- =============================================
-- Step 4: Create Stored Procedure for Batch Incremental Refresh
-- =============================================

IF OBJECT_ID('dev.SP_Refresh_MT_FCT_COMBINED_HEALTH_CLIENT_TABLE_BatchIncrementalRefresh', 'P') IS NOT NULL
    DROP PROCEDURE dev.SP_Refresh_MT_FCT_COMBINED_HEALTH_CLIENT_TABLE_BatchIncrementalRefresh;
GO

CREATE PROCEDURE dev.SP_Refresh_MT_FCT_COMBINED_HEALTH_CLIENT_TABLE_BatchIncrementalRefresh
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @BatchSize INT = 100000;
    DECLARE @RowsInserted INT = 1;

    PRINT 'Starting batch incremental refresh of dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE...';

    WHILE (@RowsInserted > 0)
    BEGIN
        BEGIN TRY
            BEGIN TRANSACTION;

            -- Insert a batch of new records
            INSERT INTO dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE (
                   STUDY_ID,
                POSTAL_CODE,
                STREET_LINE,
                EFF_DATE,
                END_DATE
            )
            SELECT TOP (@BatchSize)
                v.STUDY_ID,
                v.POSTAL_CODE,
                v.STREET_LINE,
                CAST(v.EFF_DATE AS DATE) AS EFF_DATE,
                CAST(v.END_DATE AS DATE) AS END_DATE
            FROM [HealthFiles_test].[dev].[FCT_COMBINED_HEALTH_CLIENT_TABLE] v
            LEFT JOIN dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE mt
                ON v.STUDY_ID = mt.STUDY_ID
                AND v.EFF_DATE = mt.EFF_DATE
                AND v.POSTAL_CODE = mt.POSTAL_CODE
                AND v.STREET_LINE = mt.STREET_LINE
            WHERE mt.STUDY_ID IS NULL;

            SET @RowsInserted = @@ROWCOUNT;

            COMMIT TRANSACTION;

            PRINT CAST(@RowsInserted AS NVARCHAR(20)) + ' rows inserted in this batch.';
        END TRY
        BEGIN CATCH
            IF @@TRANCOUNT > 0
                ROLLBACK TRANSACTION;

            PRINT 'An error occurred during batch incremental refresh.';
            PRINT ERROR_MESSAGE();

            -- Exit the loop in case of error
            BREAK;
        END CATCH
    END

    PRINT 'Batch incremental refresh completed.';
END
GO

PRINT 'Stored Procedure dev.SP_Refresh_MT_FCT_COMBINED_HEALTH_CLIENT_TABLE_BatchIncrementalRefresh created successfully.';


d. Schedule the Stored Procedure

In [ ]:
EXEC dev.SP_Refresh_MT_FCT_COMBINED_HEALTH_CLIENT_TABLE_BatchIncrementalRefresh;


c. Monitoring and Maintenance

Regular Monitoring: Continuously monitor the performance of the materialized table and the incremental refresh process. Use SQL Server's Dynamic Management Views (DMVs) and Azure Data Studio’s Monitoring Tools for insights.

  

Index Maintenance: Regularly rebuild or reorganize indexes to prevent fragmentation.

In [ ]:
-- Rebuild Indexes
ALTER INDEX ALL ON dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE REBUILD;

-- Or Reorganize if fragmentation is low
-- ALTER INDEX ALL ON dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE REORGANIZE;


Update Statistics: Keep statistics up-to-date to ensure the query optimizer makes informed decisions.

In [ ]:
-- Update statistics for the materialized table
UPDATE STATISTICS dev.MT_FCT_COMBINED_HEALTH_CLIENT_TABLE;


6\. Final Adjustments to Your Processing Script

With the materialized table in place and the incremental refresh process set up, update your downstream processing scripts to reference dev.MT\_FCT\_COMBINED\_HEALTH\_CLIENT\_TABLE instead of the view. Ensure that your scripts account for the RecordID and handle multiple records per (STUDY\_ID, EFF\_DATE, POSTAL\_CODE, STREET\_LINE) appropriately.

  

Example Adjusted Processing Script Snippet

In [ ]:
-- =============================================
-- Step 0: Initialize Timing Variables
-- =============================================

DECLARE @StartTime DATETIME2, @EndTime DATETIME2, @DurationSeconds FLOAT;

-- =============================================
-- Step 1: Clean and Prepare Data (Using Staging Table)
-- =============================================

SET @StartTime = SYSDATETIME();

-- Drop temporary tables if they already exist
IF OBJECT_ID('tempdb..#CleanedData') IS NOT NULL DROP TABLE #CleanedData;

-- Select and transform data from the staging table
SELECT
    STUDY_ID,
    POSTAL_CODE,
    STREET_LINE,
    -- Extract the first two words from STREET_LINE
    CASE 
        WHEN STREET_LINE IS NOT NULL THEN
            CASE
                WHEN CHARINDEX(' ', STREET_LINE) > 0 THEN
                    LEFT(
                        STREET_LINE,
                        CHARINDEX(' ', STREET_LINE + ' ', CHARINDEX(' ', STREET_LINE + ' ') + 1) - 1
                    )
                ELSE STREET_LINE
            END
        ELSE STREET_LINE
    END AS STREET_FIRST_TWO_WORDS,
    EFF_DATE,
    END_DATE
INTO #CleanedData
FROM [HealthFiles_test].[dev].[Staging_FCT_COMBINED_HEALTH_CLIENT_TABLE]
WHERE STUDY_ID = '001A9042FA9A48CC00E28DD031448F25B9CE08AA676282D6'; -- Replace with your STUDY_ID

-- Create indexes to optimize subsequent operations
CREATE NONCLUSTERED INDEX IX_CleanedData_EFF_DATE ON #CleanedData (STUDY_ID, EFF_DATE);

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 1: Clean and Prepare Data took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';

-- =============================================
-- Step 2: Compute Previous Values Using Window Functions
-- =============================================

SET @StartTime = SYSDATETIME();

-- Create and populate #LaggedData with previous postal code and street words
IF OBJECT_ID('tempdb..#LaggedData') IS NOT NULL DROP TABLE #LaggedData;

SELECT
    CD.STUDY_ID,
    CD.POSTAL_CODE,
    CD.STREET_LINE,
    CD.STREET_FIRST_TWO_WORDS,
    CD.EFF_DATE,
    CD.END_DATE,
    LAG(CD.POSTAL_CODE) OVER (PARTITION BY CD.STUDY_ID ORDER BY CD.EFF_DATE) AS Prev_POSTAL_CODE,
    LAG(CD.STREET_FIRST_TWO_WORDS) OVER (PARTITION BY CD.STUDY_ID ORDER BY CD.EFF_DATE) AS Prev_STREET_FIRST_TWO_WORDS
INTO #LaggedData
FROM #CleanedData CD
ORDER BY CD.EFF_DATE;

-- Create index to optimize window functions
CREATE NONCLUSTERED INDEX IX_LaggedData_EFF_DATE ON #LaggedData (EFF_DATE);

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 2: Compute Previous Values Using Window Functions took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';

-- =============================================
-- Step 3: Flag Changes in Address
-- =============================================

SET @StartTime = SYSDATETIME();

-- Create and populate #ChangeFlagData with ChangeFlag
IF OBJECT_ID('tempdb..#ChangeFlagData') IS NOT NULL DROP TABLE #ChangeFlagData;

SELECT
    LD.STUDY_ID,
    LD.POSTAL_CODE,
    LD.STREET_LINE,
    LD.STREET_FIRST_TWO_WORDS,
    LD.EFF_DATE,
    LD.END_DATE,
    LD.Prev_POSTAL_CODE,
    LD.Prev_STREET_FIRST_TWO_WORDS,
    CASE 
        WHEN LD.Prev_POSTAL_CODE IS NULL 
             OR LD.Prev_STREET_FIRST_TWO_WORDS IS NULL 
             OR LD.Prev_POSTAL_CODE != LD.POSTAL_CODE 
             OR LD.Prev_STREET_FIRST_TWO_WORDS != LD.STREET_FIRST_TWO_WORDS 
        THEN 1 
        ELSE 0 
    END AS ChangeFlag
INTO #ChangeFlagData
FROM #LaggedData LD
ORDER BY LD.EFF_DATE;

-- Create index to optimize subsequent operations
CREATE NONCLUSTERED INDEX IX_ChangeFlagData_EFF_DATE ON #ChangeFlagData (EFF_DATE);

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 3: Flag Changes in Address took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';

-- =============================================
-- Step 4: Assign GroupAddressKey Using Cumulative Sum
-- =============================================

SET @StartTime = SYSDATETIME();

-- Create and populate #GroupedKeyData with GroupAddressKey
IF OBJECT_ID('tempdb..#GroupedKeyData') IS NOT NULL DROP TABLE #GroupedKeyData;

SELECT
    CFD.STUDY_ID,
    CFD.POSTAL_CODE,
    CFD.STREET_LINE,
    CFD.STREET_FIRST_TWO_WORDS,
    CFD.EFF_DATE,
    CFD.END_DATE,
    CFD.Prev_POSTAL_CODE,
    CFD.Prev_STREET_FIRST_TWO_WORDS,
    CFD.ChangeFlag,
    -- Cumulative sum to assign GroupAddressKey
    SUM(CASE WHEN CFD.ChangeFlag = 1 THEN 1 ELSE 0 END) OVER (ORDER BY CFD.EFF_DATE ROWS UNBOUNDED PRECEDING) AS GroupAddressKey
INTO #GroupedKeyData
FROM #ChangeFlagData CFD
ORDER BY CFD.EFF_DATE;

-- Create index to optimize aggregation
CREATE NONCLUSTERED INDEX IX_GroupedKeyData_GroupAddressKey ON #GroupedKeyData (GroupAddressKey);

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 4: Assign GroupAddressKey Using Cumulative Sum took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';

-- =============================================
-- Step 5: Aggregate Grouped Data
-- =============================================

SET @StartTime = SYSDATETIME();

-- Create and populate #GroupedData with aggregated information
IF OBJECT_ID('tempdb..#GroupedData') IS NOT NULL DROP TABLE #GroupedData;

SELECT
    GK.STUDY_ID,
    GK.POSTAL_CODE,
    GK.STREET_FIRST_TWO_WORDS,
    MAX(GK.STREET_LINE) AS STREET_LINE,
    MIN(GK.EFF_DATE) AS EFF_DATE,
    MAX(GK.END_DATE) AS END_DATE,
    GK.GroupAddressKey
INTO #GroupedData
FROM #GroupedKeyData GK
GROUP BY GK.STUDY_ID, GK.POSTAL_CODE, GK.STREET_FIRST_TWO_WORDS, GK.GroupAddressKey;

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 5: Aggregate Grouped Data took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';

-- =============================================
-- Step 6: Insert Aggregated Data into Target Table (Optimized Join)
-- =============================================



SET @StartTime = SYSDATETIME();

-- Step 6.1: Delete existing records for STUDY_IDs present in #GroupedData
DELETE D
FROM DEV.FCT_HEALTH_CLIENT_ADDRESS_DATE D
INNER JOIN #GroupedData GD
    ON D.STUDY_ID = GD.STUDY_ID;
PRINT 'Existing records for STUDY_IDs have been deleted from DEV.FCT_HEALTH_CLIENT_ADDRESS_DATE.';


-- Insert the processed data into the target table
INSERT INTO DEV.FCT_HEALTH_CLIENT_ADDRESS_DATE (
    STUDY_ID,
    GroupAddressKey,
    POSTAL_CODE,
    STREET_LINE,
    EFF_DATE,
    END_DATE,
    CITY,
    LATITUDE,
    LONGITUDE
)
SELECT 
    GD.STUDY_ID,
    GD.GroupAddressKey,
    GD.POSTAL_CODE,
    GD.STREET_LINE,
    GD.EFF_DATE,
    GD.END_DATE,       
    B.[CITY], 
    B.LATITUDE, 
    B.LONGITUDE
FROM #GroupedData GD
LEFT JOIN dev.DIM_HEALTH_CLIENT_ADDRESS B
    ON GD.POSTAL_CODE = B.POSTAL_CODE 
    AND GD.STREET_LINE = B.STREET_LINE;

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 6: Insert Aggregated Data into Target Table took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';

-- =============================================
-- Step 7: Cleanup Temporary Tables
-- =============================================

SET @StartTime = SYSDATETIME();

DROP TABLE IF EXISTS #CleanedData, #LaggedData, #ChangeFlagData, #GroupedKeyData, #GroupedData;

SET @EndTime = SYSDATETIME();
SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
PRINT 'Step 7: Cleanup Temporary Tables took ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';


Assuming you've materialized the view into a staging table and optimized the indexing, here's an optimized version of your query with additional refinements:

 Advanced Optimization Techniques

a. Use Covering Indexes

Ensure that all columns used in the SELECT, JOIN, WHERE, and ORDER BY clauses are included in the indexes. Covering indexes can prevent SQL Server from having to perform additional lookups.

  

b. Batch Inserts and Avoid Lock Escalation

For large insert operations, consider inserting data in smaller batches to avoid long transactions that can cause lock escalations and excessive tempdb usage.

Note: This example assumes that STUDY\_ID is sequential and can be used to paginate through data. Adjust the logic as necessary based on your data characteristics.

In [ ]:
-- =============================================
-- Step 0: Initialize Timing Variables
-- =============================================

DECLARE @StartTime DATETIME2, @EndTime DATETIME2, @DurationSeconds FLOAT;
DECLARE @BatchSize INT = 100; -- Adjust the batch size as needed
DECLARE @ProcessedCount INT = 0;
DECLARE @TotalCount INT;

-- =============================================
-- Step 1: Initialize the StudyID List
-- =============================================

-- Create a temporary table to hold STUDY_IDs to process
IF OBJECT_ID('tempdb..#StudyIDList') IS NOT NULL DROP TABLE #StudyIDList;

SELECT STUDY_ID
INTO #StudyIDList
FROM dev.DIM_HEALTH_CLIENT_ID;

-- Get the total number of STUDY_IDs to process
SELECT @TotalCount = COUNT(*) FROM #StudyIDList;

PRINT 'Total STUDY_IDs to process: ' + CAST(@TotalCount AS NVARCHAR);

-- =============================================
-- Step 2: Batch Processing Loop
-- =============================================

WHILE EXISTS (SELECT 1 FROM #StudyIDList)
BEGIN
    BEGIN TRY
        BEGIN TRANSACTION;

        -- Initialize Timing for the Batch
        SET @StartTime = SYSDATETIME();

        -- Select the top @BatchSize STUDY_IDs for this batch
        SELECT TOP (@BatchSize) STUDY_ID
        INTO #CurrentBatch
        FROM #StudyIDList;

        -- =============================================
        -- Step 3: Clean and Prepare Data (Using Staging Table)
        -- =============================================

        -- Drop temporary tables if they already exist
        IF OBJECT_ID('tempdb..#CleanedData') IS NOT NULL DROP TABLE #CleanedData;

        -- Select and transform data from the staging table for the current batch
        SELECT
            S.STUDY_ID,
            POSTAL_CODE,
            STREET_LINE,
            -- Extract the first two words from STREET_LINE
            CASE 
                WHEN STREET_LINE IS NOT NULL THEN
                    CASE
                        WHEN CHARINDEX(' ', STREET_LINE) > 0 THEN
                            LEFT(
                                STREET_LINE,
                                CHARINDEX(' ', STREET_LINE + ' ', CHARINDEX(' ', STREET_LINE + ' ') + 1) - 1
                            )
                        ELSE STREET_LINE
                    END
                ELSE STREET_LINE
            END AS STREET_FIRST_TWO_WORDS,
            CAST(EFF_DATE AS DATE) AS EFF_DATE,
            CAST(END_DATE AS DATE) AS END_DATE
        INTO #CleanedData
        FROM [HealthFiles_test].[dev].[MT_FCT_COMBINED_HEALTH_CLIENT_TABLE] S
        INNER JOIN #CurrentBatch C
            ON S.STUDY_ID = C.STUDY_ID;

        -- Create indexes to optimize subsequent operations
        CREATE NONCLUSTERED INDEX IX_CleanedData_EFF_DATE ON #CleanedData (STUDY_ID, EFF_DATE);
        
        -- =============================================
        -- Step 4: Compute Previous Values Using Window Functions
        -- =============================================

        -- Drop temporary tables if they already exist
        IF OBJECT_ID('tempdb..#LaggedData') IS NOT NULL DROP TABLE #LaggedData;

        -- Create and populate #LaggedData with previous postal code and street words
        SELECT
            CD.STUDY_ID,
            CD.POSTAL_CODE,
            CD.STREET_LINE,
            CD.STREET_FIRST_TWO_WORDS,
            CD.EFF_DATE,
            CD.END_DATE,
            CD.RecordID,
            LAG(CD.POSTAL_CODE) OVER (PARTITION BY CD.STUDY_ID ORDER BY CD.EFF_DATE, END_DATE, CD.RecordID) AS Prev_POSTAL_CODE,
            LAG(CD.STREET_FIRST_TWO_WORDS) OVER (PARTITION BY CD.STUDY_ID ORDER BY CD.EFF_DATE,, END_DATE, CD.RecordID) AS Prev_STREET_FIRST_TWO_WORDS
        INTO #LaggedData
        FROM (
            SELECT 
                *, 
                ROW_NUMBER() OVER (PARTITION BY STUDY_ID ORDER BY EFF_DATE, END_DATE) AS RecordID
            FROM #CleanedData
        ) CD
        ORDER BY CD.EFF_DATE, CD.RecordID;

        -- Create index to optimize window functions
        CREATE NONCLUSTERED INDEX IX_LaggedData_EFF_DATE ON #LaggedData (EFF_DATE);

        -- =============================================
        -- Step 5: Flag Changes in Address
        -- =============================================

        -- Drop temporary tables if they already exist
        IF OBJECT_ID('tempdb..#ChangeFlagData') IS NOT NULL DROP TABLE #ChangeFlagData;

        -- Create and populate #ChangeFlagData with ChangeFlag
        SELECT
            LD.STUDY_ID,
            LD.POSTAL_CODE,
            LD.STREET_LINE,
            LD.STREET_FIRST_TWO_WORDS,
            LD.EFF_DATE,
            LD.END_DATE,
            LD.Prev_POSTAL_CODE,
            LD.Prev_STREET_FIRST_TWO_WORDS,
            LD.RecordID,
            CASE 
                WHEN LD.Prev_POSTAL_CODE IS NULL 
                     OR LD.Prev_STREET_FIRST_TWO_WORDS IS NULL 
                     OR LD.Prev_POSTAL_CODE != LD.POSTAL_CODE 
                     OR LD.Prev_STREET_FIRST_TWO_WORDS != LD.STREET_FIRST_TWO_WORDS 
                THEN 1 
                ELSE 0 
            END AS ChangeFlag
        INTO #ChangeFlagData
        FROM #LaggedData LD
        ORDER BY LD.EFF_DATE, LD.RecordID;

        -- Create index to optimize subsequent operations
        CREATE NONCLUSTERED INDEX IX_ChangeFlagData_EFF_DATE ON #ChangeFlagData (EFF_DATE);

        -- =============================================
        -- Step 6: Assign GroupAddressKey Using Cumulative Sum
        -- =============================================

        -- Drop temporary tables if they already exist
        IF OBJECT_ID('tempdb..#GroupedKeyData') IS NOT NULL DROP TABLE #GroupedKeyData;

        -- Create and populate #GroupedKeyData with GroupAddressKey
        SELECT
            CFD.STUDY_ID,
            CFD.POSTAL_CODE,
            CFD.STREET_LINE,
            CFD.STREET_FIRST_TWO_WORDS,
            CFD.EFF_DATE,
            CFD.END_DATE,
            CFD.Prev_POSTAL_CODE,
            CFD.Prev_STREET_FIRST_TWO_WORDS,
            CFD.ChangeFlag,
            -- Cumulative sum to assign GroupAddressKey
            SUM(CASE WHEN CFD.ChangeFlag = 1 THEN 1 ELSE 0 END) OVER (PARTITION BY CFD.STUDY_ID ORDER BY CFD.EFF_DATE, CFD.RecordID ROWS UNBOUNDED PRECEDING) AS GroupAddressKey
        INTO #GroupedKeyData
        FROM #ChangeFlagData CFD
        ORDER BY CFD.EFF_DATE, CFD.RecordID;

        -- Create index to optimize aggregation
        CREATE NONCLUSTERED INDEX IX_GroupedKeyData_GroupAddressKey ON #GroupedKeyData (GroupAddressKey);

        -- =============================================
        -- Step 7: Aggregate Grouped Data
        -- =============================================

        -- Drop temporary tables if they already exist
        IF OBJECT_ID('tempdb..#GroupedData') IS NOT NULL DROP TABLE #GroupedData;

        -- Create and populate #GroupedData with aggregated information
        SELECT
            GK.STUDY_ID,
            GK.POSTAL_CODE,
            GK.STREET_FIRST_TWO_WORDS,
            MAX(GK.STREET_LINE) AS STREET_LINE,
            MIN(GK.EFF_DATE) AS EFF_DATE,
            MAX(GK.END_DATE) AS END_DATE,
            GK.GroupAddressKey
        INTO #GroupedData
        FROM #GroupedKeyData GK
        GROUP BY GK.STUDY_ID, GK.POSTAL_CODE, GK.STREET_FIRST_TWO_WORDS, GK.GroupAddressKey;

        -- =============================================
        -- Step 8: Insert Aggregated Data into Target Table (Optimized Join)
        -- =============================================

        -- Insert the processed data into the target table
        INSERT INTO DEV.FCT_HEALTH_CLIENT_ADDRESS_DATE (
            STUDY_ID,
            GroupAddressKey,
            POSTAL_CODE,
            STREET_LINE,
            EFF_DATE,
            END_DATE,
            CITY,
            LATITUDE,
            LONGITUDE
        )
        SELECT 
            GD.STUDY_ID,
            GD.GroupAddressKey,
            GD.POSTAL_CODE,
            GD.STREET_LINE,
            GD.EFF_DATE,
            GD.END_DATE,       
            B.[CITY], 
            B.LATITUDE, 
            B.LONGITUDE
        FROM #GroupedData GD
        LEFT JOIN dev.DIM_HEALTH_CLIENT_ADDRESS B
            ON GD.POSTAL_CODE = B.POSTAL_CODE 
            AND GD.STREET_LINE = B.STREET_LINE;

        -- =============================================
        -- Step 9: Cleanup Temporary Tables for the Batch
        -- =============================================
        SET @ProcessedCount = @ProcessedCount + (SELECT COUNT(*) FROM #CurrentBatch);
        DROP TABLE IF EXISTS #CleanedData, #LaggedData, #ChangeFlagData, #GroupedKeyData, #GroupedData, #CurrentBatch;

        -- Calculate Duration for the Batch
        SET @EndTime = SYSDATETIME();
        SET @DurationSeconds = DATEDIFF(SECOND, @StartTime, @EndTime);
        
        PRINT 'Processed ' + CAST(@ProcessedCount AS NVARCHAR) + ' out of ' + CAST(@TotalCount AS NVARCHAR) + ' STUDY_IDs. Current Batch Duration: ' + CAST(@DurationSeconds AS NVARCHAR) + ' seconds.';
         
        COMMIT TRANSACTION;
    END TRY
    BEGIN CATCH
        IF @@TRANCOUNT > 0
            ROLLBACK TRANSACTION;

        PRINT 'An error occurred during batch processing.';
        PRINT ERROR_MESSAGE();

        -- Optionally, you can decide to exit the loop or continue
        BREAK;
    END CATCH
END

PRINT 'All batches processed successfully.';
